# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring, built on top of a binary classifier.**

The decision itself ("which pages first") is inherently a ranking question, not a plain yes/no --
an editor works down an ordered list until their capacity runs out. The cleanest way to produce
that ranking with real evidence behind it is to train a classifier that estimates
`P(page is currently declining)`, then sort pages by that probability (optionally blended with a
transparent baseline score, the way `scripts/04_evaluate_and_export.py` does). So: a classification
model underneath, a ranking/scoring deliverable on top.


In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("is_declining_label is binary (0/1) -> supports a classifier.")
print("The actual deliverable is the RANKING of P(decline=1) across pages,")
print("not the raw 0/1 label by itself -- that's why this is ranking/scoring, not plain classification.")


is_declining_label is binary (0/1) -> supports a classifier.
The actual deliverable is the RANKING of P(decline=1) across pages,
not the raw 0/1 label by itself -- that's why this is ranking/scoring, not plain classification.


## 2. Target or proxy

**Target:** `is_declining_label = 1` when `trend_direction == "down"`, i.e. impressions in the
last 30 days fell more than 20% versus the prior 30 days.

**Where it comes from:** a *defined rule* -- a fixed -20% threshold applied to `trend_pct` -- not a
fully independent, later-observed outcome. That makes it a **proxy label**, not the real thing
FlyRank ultimately cares about (does the page keep declining, and does review actually help?). It's
an honest, simple starting proxy for these first weeks; a stronger capstone version would instead
use `prior 90 days of features -> decline over the NEXT 30 days`, a genuinely future window. Flagging
that limitation now, in writing, rather than discovering it later.


In [2]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("trend_pct summary (the number the label buckets are computed from):")
print(df["trend_pct"].describe().round(2))
print("\ntrend_direction value counts:")
print(df["trend_direction"].value_counts())
print("\n-> trend_direction and trend_pct are excluded from every feature list precisely")
print("   because the label is computed FROM them (see Week-3 leakage check for the proof).")


trend_pct summary (the number the label buckets are computed from):
count    26612.00
mean        -4.79
std        473.86
min       -100.00
25%        -62.60
50%        -33.50
75%          0.00
max      44900.00
Name: trend_pct, dtype: float64

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

-> trend_direction and trend_pct are excluded from every feature list precisely
   because the label is computed FROM them (see Week-3 leakage check for the proof).


## 3. Success metric

**Metric: Precision@50.** Of the top 50 pages the ranking recommends reviewing first, what
fraction are actually declining? 50 matches a realistic weekly review capacity for one client (see
the Week-1 notebook), so it directly measures what an editor experiences: "if I only get through
the top of this list, was my time well spent?" I'll track ROC-AUC and average precision alongside
it as sanity checks, but Precision@50 is the number I would defend to a non-technical stakeholder.


In [3]:
import json
res = json.load(open("outputs/model_results.json"))
print("Baseline (hand rule)  Precision@50:", round(res["baseline"]["baseline_precision_at_50"], 3))
print("Random forest         Precision@50:", round(res["models"]["random_forest"]["precision_at_50"], 3))
print("(from a fresh `python scripts/run_all.py` run -- these are the numbers I'm trying to beat")
print(" and the numbers I'm trying to explain, not numbers copied from the docs.)")


Baseline (hand rule)  Precision@50: 0.24
Random forest         Precision@50: 0.68
(from a fresh `python scripts/run_all.py` run -- these are the numbers I'm trying to beat
 and the numbers I'm trying to explain, not numbers copied from the docs.)


## 4. The unit of analysis, as a real dataframe

One row = one content page (`content_id`), aggregated over its trailing 90-day window as of the
export date. Shown below as the actual feature columns my lane's model will consume.


In [4]:
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

# the log_* features are engineered in the prep step (scripts/01_prepare_features.py), not raw columns
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))

unit_cols = ["content_id", "client_id"] + MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
print("One row = one content page. Duplicate content_id rows:", df["content_id"].duplicated().sum())
df[unit_cols].head(3)


One row = one content page. Duplicate content_id rows: 0


,content_id,client_id,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,...,scroll_rate,ai_traffic_pct,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,content_304f48230142,client_f369cb89fc,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,...,4.55,0.0,HIGH,keyword article,transactional,181-365,0-30,2000-3500,good,striking
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,...,10.00,0.0,LOW,keyword article,informational,365+,0-30,2000-3500,good,page_3_5
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,...,28.57,0.0,LOW,keyword article,informational,91-180,0-30,3500+,good,page_3_5


## 5. Why ML beats a fixed rule here

A one-line hand rule like "stale AND visible" only uses 2 of the ~25 available signals, and it
can't weigh them -- a page that's borderline-stale but hugely visible gets the same 0 as a page
that's neither. The signals here are individually weak on their own (Week-1 finding: `search_volume`
barely correlates with real traffic) but numerous, and they interact -- position, freshness, CTR,
and content type all move the outcome differently depending on each other. That is exactly the
"many, tangled, shifting signals" case where a fixed if-statement runs out of room and a model
earns its place. The gap below is the evidence, not a guess.


In [5]:
print(f"Hand rule      Precision@50: {res['baseline']['baseline_precision_at_50']:.3f}")
print(f"Random forest  Precision@50: {res['models']['random_forest']['precision_at_50']:.3f}")
lift = res['models']['random_forest']['precision_at_50'] / res['baseline']['baseline_precision_at_50']
print(f"-> ~{lift:.1f}x lift -- one rule using 2 signals vs a model weighing ~25 at once.\n")

print("Top 5 signals the model actually leaned on (feature_importance_top):")
for row in res["best_model"]["feature_importance_top"][:5]:
    print(f"  {row['feature']:<28} {row['importance']:.3f}")


Hand rule      Precision@50: 0.240
Random forest  Precision@50: 0.680
-> ~2.8x lift -- one rule using 2 signals vs a model weighing ~25 at once.

Top 5 signals the model actually leaned on (feature_importance_top):
  days_with_impressions        0.160
  log_impressions_90d          0.128
  avg_position                 0.108
  content_age_days             0.095
  word_count                   0.041


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.